# Exercise: Contextual String Embeddings for Sequence Labeling (Akbik et al., 2018)

In the seminar we covered Flair contextual string embeddings and the BiLSTM-CRF sequence labeler.
This short exercise focuses on understanding the architecture and implementing the CRF layer.


In [3]:
def TODO(todo: str = "Fill the blank"):
    raise ValueError(todo)

## Exercise Part 1: Akbik et al. recap and CRF


### Task 1.1

Read the paper of Akbik et al. Then answer briefly:

1. What is the key difference between static word embeddings (e.g. GloVe) and contextual string embeddings?
2. Why is a BiLSTM used also as sequence tagger on top of the contextual word embeddings? How does its performance compare to a standard feedforward network?
3. What problem does a Conditional Random Field (CRF; eq. 10 & 11) solve that independent token classification does not?


--- 

### Task 1.2 
Assume a BiLSTM produced the following emission scores.


In [ ]:
TOKENS = ["George", "Lewis", "lives", "in", "Berlin"]

EMISSIONS = [
    {"B-PER": 5, "I-PER": 4, "O": 1},   # George
    {"B-PER": 2, "I-PER": 1, "O": 0},   # Lewis
    {"B-LOC": 1, "I-LOC": 1, "O": 5},   # lives
    {"B-LOC": 2, "I-LOC": 1, "O": 6},   # in
    {"B-LOC": 6, "I-LOC": 1, "O": 2},   # Berlin
]

#### Question

If we classify each token independently, which labels would be chosen?

### Task 1.3

The CRF additionally learns transition scores. Assume the following scores were learned by the model:


In [ ]:
TRANSITIONS = {
    ("I-PER", "B-PER"): -20,
    ("B-PER", "B-PER"): -3,
    ("B-PER", "I-PER"): 1,
    ("B-PER", "O"): 0,
    ("O", "O"): 0,
    ("O", "B-LOC"): 2,
    ("B-LOC", "O"): 0,
    ("B-LOC", "B-LOC"): -3,
}

The score of a sequence is:
$$ \text{score} =
\text{sum of emission scores}
+
\text{sum of transition scores}$$

Complete the implementation. Then compare the candidate sequences.

In [9]:
def sequence_score(tags, emissions, transitions):
    score = emissions[0][tags[0]]

    for i in range(1, len(tags)):
        score += TODO("Add emission score")
        score += TODO("Add transition score")

    return score

In [ ]:
candidate_1 = ["B-PER", "O", "O", "O", "B-LOC"]

candidate_2 = ["B-PER", "B-PER", "O", "O", "B-LOC"]

candidate_3 = ["B-PER", "I-PER", "O", "O", "B-LOC"]

for candidate in [candidate_1, candidate_2, candidate_3]:
    print(f"Candidate: {candidate}, Score: {sequence_score(candidate, EMISSIONS, TRANSITIONS)}")

#### Questions
- Which sequence receives the highest score?
- Why is candidate_2 penalized?

# Part 2: CRF in Pytorch

Given emission scores $\mathbf{e} \in \mathbb{R}^{T \times K}$ from the BiLSTM and a transition matrix $\mathbf{A} \in \mathbb{R}^{K \times K}$, the score of a tag sequence $\mathbf{y} = (y_0, \dots, y_{T-1})$ is

$$s(\mathbf{y}) = \sum_{t=0}^{T-1} e_t[y_t] \;+\; \sum_{t=1}^{T-1} A[y_{t-1},\, y_t]$$

#### Task 2.1: 

Finish below pytorch implementation of a CRF scorer. Initialize the transition matrix randomly and make sure it can be trained (Hint: Check pytorch documentation for nn.Parameter)

In [ ]:
import torch
import torch.nn as nn

# ── CRF (Eq. 10-11) ───────────────────────────────────────────────────────
class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags    = num_tags
        self.transitions = TODO("Initialize transition scores")

    def score(self, emit_scores, tags):
        score = emit_scores[0, tags[0]]
        for t in range(1, emit_scores.size(0)):
            score += TODO("Add emission score") + TODO("Add transition score")
        return score
